In [1]:
# libraries
import os
import json
import time
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import nbformat
from IPython.display import display

In [2]:
# globals; run through run_pipeline.ipynb
ROOT = Path(os.environ['NCA_PROJECT_ROOT'])
TASK = ROOT / 'd_phase1_train'
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CFG = json.loads((INPUT / 'config.json').read_text())
DEVICE = torch.device(CFG['device'])
torch.set_num_threads(CFG['cpu_threads'])
torch.use_deterministic_algorithms(True)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

In [3]:
# functions passed through c_models/output; definitions only, no training side effects
runtime_nb = nbformat.read(INPUT / 'runtime.ipynb', as_version=4)
for runtime_cell in runtime_nb.cells:
    exec(compile(runtime_cell.source, 'runtime.ipynb', 'exec'), globals())
PACK = torch.load(INPUT / 'dataset.pt', weights_only=True, map_location='cpu')
SPECS = CFG['models']

In [4]:
# content identities for exact checkpoint resumption
data_hash = hashlib.sha256((INPUT / 'dataset.pt').read_bytes()).hexdigest()
code_hash = (INPUT / 'runtime.sha256').read_text()
run_rows = []

In [5]:
# train Phase 1; identical condition/augmentation schedules within each paired replicate
runs = OUTPUT / 'phase1_runs'
runs.mkdir(exist_ok=True)
for regime in CFG['budget_regimes']:
    for replicate in range(CFG['replicates']):
        for spec in SPECS:
            name = spec['name']
            run_name = f'{regime}__{name}__r{replicate}'
            warm_start = None
            result = fit_run(spec, CFG, PACK, 1, regime, replicate, runs / run_name,
                             data_hash, code_hash, warm_start=warm_start)
            run_rows.append(result)
            pd.DataFrame(run_rows).to_csv(OUTPUT / 'phase1_manifest.csv', index=False)
display(pd.DataFrame(run_rows))

,step,train_loss,val_loss,grad_norm,switches_this_batch,total_bits,total_cell_updates,seconds,phase,model,regime,replicate
0,2,0.12817,0.128057,0.21508,0,69184,9216,0.048466,1,1A_det,updates,0


,step,train_loss,val_loss,grad_norm,switches_this_batch,total_bits,total_cell_updates,seconds,phase,model,regime,replicate
0,4,0.129514,0.123413,0.311509,0,155664,20736,0.101987,1,1A_det,updates,0


,step,train_loss,val_loss,grad_norm,switches_this_batch,total_bits,total_cell_updates,seconds,phase,model,regime,replicate
0,2,0.128692,0.129665,0.162469,0,69184,9216,0.035645,1,1A_bern,updates,0


,step,train_loss,val_loss,grad_norm,switches_this_batch,total_bits,total_cell_updates,seconds,phase,model,regime,replicate
0,4,0.132428,0.126565,0.239902,0,155664,20736,0.08006,1,1A_bern,updates,0


,step,train_loss,val_loss,grad_norm,switches_this_batch,total_bits,total_cell_updates,seconds,phase,model,regime,replicate
0,2,0.129265,0.131375,0.126204,0,138368,9216,0.055873,1,1B_det,updates,0


,step,train_loss,val_loss,grad_norm,switches_this_batch,total_bits,total_cell_updates,seconds,phase,model,regime,replicate
0,4,0.135725,0.130095,0.191206,0,311328,20736,0.133927,1,1B_det,updates,0


... (15 outputs left out)

,step,train_loss,val_loss,grad_norm,switches_this_batch,total_bits,total_cell_updates,seconds,phase,model,regime,replicate
0,4,0.136848,0.13128,0.100053,0,155664,10368,0.092359,1,1B_rbern,bits,0


,step,train_loss,val_loss,grad_norm,switches_this_batch,total_bits,total_cell_updates,seconds,phase,model,regime,replicate
0,2,0.129393,0.131762,0.071218,0,69184,4608,0.032444,1,1B_qrbern,bits,0


,step,train_loss,val_loss,grad_norm,switches_this_batch,total_bits,total_cell_updates,seconds,phase,model,regime,replicate
0,4,0.136418,0.13077,0.109863,0,155664,10368,0.079244,1,1B_qrbern,bits,0


,phase,model,family,regime,replicate,parameters,width,best_step,best_val_loss,signature,optimizer_steps,training_bits,training_cell_updates,seconds,checkpoint
0,1,1A_det,message,updates,0,1219,22,4,0.123413,94736490a0dc383b4cf3e62cb593b64080c35dfe1401a4...,4,155664,20736,0.112913,updates__1A_det__r0/best.pt
1,1,1A_bern,message,updates,0,1219,22,4,0.126565,a4fb07b846b9686a26672aea08af200c8d2497c9448e72...,4,155664,20736,0.087849,updates__1A_bern__r0/best.pt
2,1,1B_det,query_answer,updates,0,1160,10,4,0.130095,be652619f51e3be0dfb2c1ddc4b2d57bc272ec757c8829...,4,311328,20736,0.143442,updates__1B_det__r0/best.pt
3,1,1B_qbern,query_answer,updates,0,1160,10,4,0.129402,25a4df8f6bdd39dd369b3663310bd299b5a5aeea592165...,4,311328,20736,0.142962,updates__1B_qbern__r0/best.pt
4,1,1B_rbern,query_answer,updates,0,1160,10,4,0.129906,79a8f88427b64d0d8f028f1661dbb681e464e57eccf4eb...,4,311328,20736,0.142700,updates__1B_rbern__r0/best.pt
5,1,1B_qrbern,query_answer,updates,0,1160,10,4,0.128922,e36adfa63b22d6c6644e578d342808b1b5d3d4578b28c0...,4,311328,20736,0.143273,updates__1B_qrbern__r0/best.pt
6,1,1A_det,message,bits,0,1219,22,4,0.123413,810a3124d019d845075bf732020ad87d79091d7e497f0b...,4,155664,20736,0.086165,bits__1A_det__r0/best.pt
7,1,1A_bern,message,bits,0,1219,22,4,0.126565,085c41b484503a84738831262f01bace0deda17529eb30...,4,155664,20736,0.085039,bits__1A_bern__r0/best.pt
8,1,1B_det,query_answer,bits,0,1160,10,4,0.131364,942fc06c7cc5b3921f702b115840609f5f32a5244a9888...,4,155664,10368,0.087093,bits__1B_det__r0/best.pt
9,1,1B_qbern,query_answer,bits,0,1160,10,4,0.131001,ae07fd44fd5c3b702111d86b0b833bcc6e2b8520098068...,4,155664,10368,0.094031,bits__1B_qbern__r0/best.pt
